# 02 — Data Wrangling : préparation et audit qualité

Ce notebook dépend du CSV produit par le notebook 01. Il construit une cible de classification binaire seulement quand l’API donne un résultat explicite. Les atterrissages inconnus ne sont **pas** traités automatiquement comme des échecs.

In [ ]:
import pandas as pd
from pathlib import Path
DATA = Path("data") if Path("data").exists() else Path("../data")
FIG = Path("figures") if Path("figures").exists() else Path("../figures")
FIG.mkdir(exist_ok=True)
file = DATA / "spacex_falcon9_api_current.csv"
if not file.exists():
    raise FileNotFoundError("Exécute d’abord 01_SpaceX_API.ipynb depuis la racine du projet ou le dossier notebooks.")
df = pd.read_csv(file)
print("Dimensions initiales :", df.shape)
display(df.head())

In [ ]:
display(df.dtypes.to_frame("Type"))
display(df.isnull().sum().to_frame("Missing values"))
print("Doublons FlightID :", df.duplicated("FlightID").sum())

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
df["Year"] = df["Date"].dt.year
# pd.read_csv peut avoir converti les booléens en objets : normalisation robuste
normalize_bool = {True: True, False: False, "True": True, "False": False, "true": True, "false": False, 1: True, 0: False}
df["LandingSuccess"] = df["LandingSuccess"].map(normalize_bool)
known = df.loc[df["LandingSuccess"].notna()].copy()
known["Class"] = known["LandingSuccess"].astype(int)
print("Lancements avec statut d’atterrissage connu :", len(known))
print("Succès (1) / échecs (0) :")
display(known["Class"].value_counts().to_frame("Count"))

In [ ]:
summary = known.groupby("LaunchSite", dropna=False).agg(launches=("Class", "size"), successful_landings=("Class", "sum"), success_rate=("Class", "mean")).reset_index().sort_values("launches", ascending=False)
display(summary)
known.to_csv(DATA / "spacex_falcon9_clean_current.csv", index=False)
summary.to_csv(DATA / "landing_summary_by_site_current.csv", index=False)
print("Fichiers générés dans", DATA.resolve())

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.5))
summary.plot.bar(x="LaunchSite", y="success_rate", ax=ax, legend=False)
ax.set_ylim(0, 1)
ax.set_ylabel("Proportion des atterrissages réussis")
ax.set_xlabel("Site de lancement")
ax.set_title("Falcon 9 : réussite d’atterrissage par site (statuts connus)")
plt.xticks(rotation=25, ha="right")
fig.tight_layout()
fig.savefig(FIG / "landing_success_by_site_current.png", dpi=160)
plt.show()

## Limites et prochaines étapes
Ce jeu de données **live** n’a pas nécessairement les mêmes variables que les fichiers de référence IBM (`Orbit`, `PayloadMass`, `Serial`, etc.). Il n’est donc pas adapté à une reproduction à l’identique du notebook de classification IBM. Pour les parties SQL, Folium, Dash et ML, nous utiliserons de préférence les datasets historiques IBM avec une provenance explicitement documentée.